In [2]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
import json
import urllib
import datetime
import subprocess
import os

# --- CONFIGURACIÓN ---
server = 'db.gabssa.app' 
database = 'BAHIA'
username = 'Usr_Canales'
password = 'Huyt&233_Trwt'
RUTA_REPOSITORIO = r'C:\Users\SISTEMAS\Documents\MAQUINA DIEGO\CARTERAS\TOTALPLAY\Dashboard_Daimler'

params = urllib.parse.quote_plus(f"DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password};")
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

def procesar_periodo(nombre_etiqueta, fecha_filtro, nombre_archivo_js):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Procesando: {nombre_etiqueta}")
    try:
        # SQL para Gestiones (Desde la fecha filtro hasta hoy)
        query_g = f"""
            SELECT usuario_id, ini, asunto FROM [BAHIA].[dbo].[daimler_interaccion] 
            WHERE ini >= '{fecha_filtro}' AND cliente <> 'DAIMLER'
        """
        # SQL para Promesas (Desde la fecha filtro hasta hoy)
        query_p = f"""
            SELECT i.usuario_id, CONCAT(p.[NOMBRE_GESTOR], ' ', p.[APELLIDO_PATERNO]) AS Nombre, i.fecha_captura
            FROM [BAHIA].[dbo].[daimler_pago] i
            INNER JOIN [BAHIA].[dbo].[VW_PADRONLIGTH] p ON i.usuario_id = p.ID_EMPLEADO
            WHERE i.fecha_captura >= '{fecha_filtro}' AND cliente <> 'DAIMLER'
        """
        
        df_g = pd.read_sql(query_g, engine)
        df_p = pd.read_sql(query_p, engine)

        # Cálculos de KPIs
        total_g = len(df_g)
        activos = max(df_g['usuario_id'].nunique(), 1)
        promedio = int(round(total_g / activos, 0))
        
        # RPC %
        estatus_ok = ['PROMESA DE PAGO', 'DICE QUE YA PAGÓ', 'NEGATIVA DE PAGO', 'ACLARACIÓN', 'RECADÓ CON FAMILIAR']
        rpc = round((df_g[df_g['asunto'].isin(estatus_ok)].shape[0] / total_g * 100), 1) if total_g > 0 else 0

        # Carrera (Top 10)
        carrera = df_p.groupby('Nombre').size().sort_values(ascending=False).head(10).reset_index(name='Cant')
        
        # Premios (Basados en el periodo actual)
        premios = {
            "efectividad": carrera.iloc[0]['Nombre'] if not carrera.empty else "---",
            "trabajador": "ID:" + str(df_g.groupby('usuario_id').size().idxmax()) if not df_g.empty else "---",
            "racha": carrera.iloc[0]['Nombre'] if not carrera.empty else "---"
        }

        # Gráfica Hora (Solo para el periodo, aunque en Semana/Mes se verá el acumulado por horas del día)
        df_g['hora'] = pd.to_datetime(df_g['ini']).dt.hour
        g_hora = df_g.groupby('hora').size().reindex(range(8, 21), fill_value=0)
        contact = df_g['asunto'].value_counts().head(8).to_dict()

        data = {
            "periodo": nombre_etiqueta,
            "kpis": {
                "total_promesas": len(df_p),
                "total_gestiones": total_g,
                "promedio_por_asesor": promedio,
                "rpc_pct": rpc
            },
            "premios": premios,
            "carrera": {"nombres": carrera['Nombre'].tolist(), "cantidades": carrera['Cant'].tolist()},
            "grafica_hora": {"horas": [f"{h}h" for h in g_hora.index], "conteo": g_hora.tolist()},
            "contactability": contact
        }

        with open(os.path.join(RUTA_REPOSITORIO, nombre_archivo_js), 'w', encoding='utf-8') as f:
            f.write(f"const datosDaimler = {json.dumps(data, indent=4, ensure_ascii=False)};")
        
        return True
    except Exception as e:
        print(f"❌ Error en {nombre_etiqueta}: {e}")
        return False

def actualizar_github():
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Sincronizando con GitHub...")
    try:
        os.chdir(RUTA_REPOSITORIO)
        subprocess.run(["git", "add", "."], check=True) # Agregamos TODO
        subprocess.run(["git", "commit", "-m", f"Triple Update: {datetime.datetime.now().strftime('%H:%M')}"], capture_output=True)
        subprocess.run(["git", "push", "origin", "main"], check=True)
        print("🚀 GitHub actualizado con éxito.")
    except Exception as e:
        print(f"ℹ️ Git: {e}")

# --- CALCULAR FECHAS ---
hoy = datetime.datetime.now()
ayer = (hoy - datetime.timedelta(days=1)).replace(hour=0, minute=0, second=0).strftime('%Y-%m-%d %H:%M:%S')
# Lunes de esta semana
lunes = (hoy - datetime.timedelta(days=hoy.weekday())).replace(hour=0, minute=0, second=0).strftime('%Y-%m-%d %H:%M:%S')
# Primero de este mes
primero_mes = hoy.replace(day=1, hour=0, minute=0, second=0).strftime('%Y-%m-%d %H:%M:%S')

# --- EJECUCIÓN ---
if __name__ == "__main__":
    p1 = procesar_periodo("DIARIO (AYER)", ayer, "datos_ayer.js")
    p2 = procesar_periodo("SEMANAL (LUNES-HOY)", lunes, "datos_semana.js")
    p3 = procesar_periodo("MENSUAL", primero_mes, "datos_mes.js")
    
    if p1 or p2 or p3:
        actualizar_github()

[12:27:30] Procesando: DIARIO (AYER)
[12:28:13] Procesando: SEMANAL (LUNES-HOY)
[12:28:49] Procesando: MENSUAL
[12:28:53] Sincronizando con GitHub...
🚀 GitHub actualizado con éxito.
